# API and demo on GPU

GPU T4, Internet on, no secret needed. Run the cells in an interactive session. The API (`/extract`, `/docs`) and the Gradio demo (`/demo`) run on the GPU and are exposed through a temporary public HTTPS URL (`https://...trycloudflare.com`). Anyone with the URL can call it while the session runs; stop the session afterwards to save GPU quota.

In [ ]:
import os
import re
import subprocess
import sys
import time
import urllib.request

REPO = "https://github.com/N-Mehdi/receipt-vlm.git"
API_LOG = "/kaggle/working/api.log"
TUNNEL_LOG = "/kaggle/working/tunnel.log"
CLOUDFLARED = (
    "https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64"
)


def run(cmd):
    """Run a shell command, stream its output and fail the notebook on error."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("command failed: " + cmd)


run("git clone --depth 1 " + REPO + " /tmp/receipt-vlm")
os.chdir("/tmp/receipt-vlm")
run("git log -1 --format=%H")
run("pip install -q -r requirements.txt && pip install -q --no-deps -e .")
# gradio needs pillow below 13.
run("pip install -q gradio==6.30.0 pillow==12.3.0")
# The preinstalled torchao is too old for peft and is not used here.
run("pip uninstall -y -q torchao")
run(
    'python -c \'import torch, transformers; print("torch", torch.__version__,'
    ' "cuda", torch.version.cuda, "transformers", transformers.__version__)\''
)

In [ ]:
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# API and demo in a separate process, on the GPU.
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.api:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open(API_LOG, "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(150):
    time.sleep(2)
    if api.poll() is not None:
        raise RuntimeError(open(API_LOG).read())
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health")
        break
    except OSError:
        pass
# The first request loads the model (a few minutes); do it here, not through the tunnel.
run("curl -s -F file=@app/examples/cord_test_1.png 'http://127.0.0.1:8000/extract?kind=cord'")

In [ ]:
# Public HTTPS URL through a Cloudflare quick tunnel (no account needed).
run("wget -q -O /tmp/cloudflared " + CLOUDFLARED + " && chmod +x /tmp/cloudflared")
tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=open(TUNNEL_LOG, "w"),
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    time.sleep(2)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    if found:
        url = found[0]
        break
if url:
    print("API docs:", url + "/docs")
    print("Demo:    ", url + "/demo")
    print("curl -F file=@invoice.jpg '" + url + "/extract?kind=invoice'")
else:
    print(open(TUNNEL_LOG).read())

In [ ]:
# Stop the tunnel and the API before ending the session.
tunnel.terminate()
api.terminate()